# Gold Layer Validation Suite

## Purpose
Comprehensive data quality validation for all **4 Gold tables** before business consumption.

## Tables Validated
1. **gold_daily_market_summary** (60 rows expected)
2. **gold_weather_impact_summary** (60 rows expected)  
3. **gold_grid_incident_summary** (~97 rows expected)
4. **gold_regional_condition_daily** (60 rows expected)

## Validation Checks
✅ **Row counts** - Match expected grain  
✅ **Null dimensions** - No null values in grain columns  
✅ **Duplicate grains** - Primary key uniqueness  
✅ **Impossible values** - No negative counts, out-of-range metrics  
✅ **Sample inspection** - Manual review of data quality

---

In [0]:
print("🔍 Starting Gold Layer Validation...")
print("Catalog: vattenfall_dev")
print("Schema: gold")
print("Tables: 4 gold tables + 3 views")

## Validation 1: gold_daily_market_summary

**Grain:** report_day × region  
**Expected:** 60 rows (15 days × 4 regions)  
**Business Logic:** Daily market price aggregations by region

In [0]:
%sql
-- Table: gold_daily_market_summary
-- Grain: report_day × region
-- Expected: 60 rows (15 days × 4 regions)

-- 1. Row count
SELECT 'Row Count' as check_name, COUNT(*) as value, 
  CASE WHEN COUNT(*) = 60 THEN '✅ PASS' ELSE '❌ FAIL' END as status
FROM vattenfall_dev.gold.gold_daily_market_summary

UNION ALL

-- 2. Null dimensions (grain columns)
SELECT 'Null report_day' as check_name, COUNT(*) as value,
  CASE WHEN COUNT(*) = 0 THEN '✅ PASS' ELSE '❌ FAIL' END as status
FROM vattenfall_dev.gold.gold_daily_market_summary
WHERE report_day IS NULL

UNION ALL

SELECT 'Null region' as check_name, COUNT(*) as value,
  CASE WHEN COUNT(*) = 0 THEN '✅ PASS' ELSE '❌ FAIL' END as status
FROM vattenfall_dev.gold.gold_daily_market_summary
WHERE region IS NULL

UNION ALL

-- 3. Duplicate grains
SELECT 'Duplicate grains' as check_name, COUNT(*) - COUNT(DISTINCT report_day, region) as value,
  CASE WHEN COUNT(*) = COUNT(DISTINCT report_day, region) THEN '✅ PASS' ELSE '❌ FAIL' END as status
FROM vattenfall_dev.gold.gold_daily_market_summary

UNION ALL

-- 4. Impossible values: negative prices
SELECT 'Negative avg_price' as check_name, COUNT(*) as value,
  CASE WHEN COUNT(*) = 0 THEN '✅ PASS' ELSE '❌ FAIL' END as status
FROM vattenfall_dev.gold.gold_daily_market_summary
WHERE avg_price_eur_mwh < 0

UNION ALL

-- 5. Impossible values: negative volumes
SELECT 'Negative volume' as check_name, COUNT(*) as value,
  CASE WHEN COUNT(*) = 0 THEN '✅ PASS' ELSE '❌ FAIL' END as status
FROM vattenfall_dev.gold.gold_daily_market_summary
WHERE total_volume_mwh < 0

UNION ALL

-- 6. Impossible values: high_price_hours out of range
SELECT 'Invalid high_price_hours' as check_name, COUNT(*) as value,
  CASE WHEN COUNT(*) = 0 THEN '✅ PASS' ELSE '❌ FAIL' END as status
FROM vattenfall_dev.gold.gold_daily_market_summary
WHERE high_price_hours < 0 OR high_price_hours > 24;

In [0]:
%sql
-- Sample rows from gold_daily_market_summary
SELECT 
  report_day,
  region,
  avg_price_eur_mwh,
  max_price_eur_mwh,
  min_price_eur_mwh,
  total_volume_mwh,
  high_price_hours,
  price_volatility_stddev
FROM vattenfall_dev.gold.gold_daily_market_summary
ORDER BY report_day DESC, region
LIMIT 10;

## Validation 2: gold_weather_impact_summary

**Grain:** report_day × region  
**Expected:** 60 rows (15 days × 4 regions)  
**Business Logic:** Daily weather risk scoring by region

In [0]:
%sql
-- Table: gold_weather_impact_summary
-- Grain: report_day × region
-- Expected: 60 rows (15 days × 4 regions)

-- 1. Row count
SELECT 'Row Count' as check_name, COUNT(*) as value, 
  CASE WHEN COUNT(*) = 60 THEN '✅ PASS' ELSE '❌ FAIL' END as status
FROM vattenfall_dev.gold.gold_weather_impact_summary

UNION ALL

-- 2. Null dimensions
SELECT 'Null report_day' as check_name, COUNT(*) as value,
  CASE WHEN COUNT(*) = 0 THEN '✅ PASS' ELSE '❌ FAIL' END as status
FROM vattenfall_dev.gold.gold_weather_impact_summary
WHERE report_day IS NULL

UNION ALL

SELECT 'Null region' as check_name, COUNT(*) as value,
  CASE WHEN COUNT(*) = 0 THEN '✅ PASS' ELSE '❌ FAIL' END as status
FROM vattenfall_dev.gold.gold_weather_impact_summary
WHERE region IS NULL

UNION ALL

-- 3. Duplicate grains
SELECT 'Duplicate grains' as check_name, COUNT(*) - COUNT(DISTINCT report_day, region) as value,
  CASE WHEN COUNT(*) = COUNT(DISTINCT report_day, region) THEN '✅ PASS' ELSE '❌ FAIL' END as status
FROM vattenfall_dev.gold.gold_weather_impact_summary

UNION ALL

-- 4. Impossible values: precipitation
SELECT 'Negative precipitation' as check_name, COUNT(*) as value,
  CASE WHEN COUNT(*) = 0 THEN '✅ PASS' ELSE '❌ FAIL' END as status
FROM vattenfall_dev.gold.gold_weather_impact_summary
WHERE total_precipitation_mm < 0

UNION ALL

-- 5. Impossible values: cloud cover out of range
SELECT 'Invalid cloud_cover' as check_name, COUNT(*) as value,
  CASE WHEN COUNT(*) = 0 THEN '✅ PASS' ELSE '❌ FAIL' END as status
FROM vattenfall_dev.gold.gold_weather_impact_summary
WHERE avg_cloud_cover_percent < 0 OR avg_cloud_cover_percent > 100

UNION ALL

-- 6. Impossible values: risk score out of range
SELECT 'Invalid weather_risk_score' as check_name, COUNT(*) as value,
  CASE WHEN COUNT(*) = 0 THEN '✅ PASS' ELSE '❌ FAIL' END as status
FROM vattenfall_dev.gold.gold_weather_impact_summary
WHERE weather_risk_score < 0 OR weather_risk_score > 9;

In [0]:
%sql
-- Sample rows from gold_weather_impact_summary
SELECT 
  report_day,
  region,
  avg_temperature_c,
  avg_wind_speed_ms,
  total_precipitation_mm,
  avg_cloud_cover_percent,
  weather_risk_score,
  weather_risk_level
FROM vattenfall_dev.gold.gold_weather_impact_summary
ORDER BY weather_risk_score DESC
LIMIT 10;

## Validation 3: gold_grid_incident_summary

**Grain:** event_day × region × severity_band  
**Expected:** ~97 rows (variable by incident occurrence)  
**Business Logic:** Incident aggregations with customer impact

In [0]:
%sql
-- Table: gold_grid_incident_summary
-- Grain: event_day × region × severity_band
-- Expected: ~97 rows (variable by incidents)

-- 1. Row count (informational)
SELECT 'Row Count' as check_name, COUNT(*) as value, '✅ INFO' as status
FROM vattenfall_dev.gold.gold_grid_incident_summary

UNION ALL

-- 2. Null dimensions
SELECT 'Null event_day' as check_name, COUNT(*) as value,
  CASE WHEN COUNT(*) = 0 THEN '✅ PASS' ELSE '❌ FAIL' END as status
FROM vattenfall_dev.gold.gold_grid_incident_summary
WHERE event_day IS NULL

UNION ALL

SELECT 'Null region' as check_name, COUNT(*) as value,
  CASE WHEN COUNT(*) = 0 THEN '✅ PASS' ELSE '❌ FAIL' END as status
FROM vattenfall_dev.gold.gold_grid_incident_summary
WHERE region IS NULL

UNION ALL

SELECT 'Null severity_band' as check_name, COUNT(*) as value,
  CASE WHEN COUNT(*) = 0 THEN '✅ PASS' ELSE '❌ FAIL' END as status
FROM vattenfall_dev.gold.gold_grid_incident_summary
WHERE severity_band IS NULL

UNION ALL

-- 3. Duplicate grains
SELECT 'Duplicate grains' as check_name, 
  COUNT(*) - COUNT(DISTINCT event_day, region, severity_band) as value,
  CASE WHEN COUNT(*) = COUNT(DISTINCT event_day, region, severity_band) THEN '✅ PASS' ELSE '❌ FAIL' END as status
FROM vattenfall_dev.gold.gold_grid_incident_summary

UNION ALL

-- 4. Impossible values: negative counts
SELECT 'Negative incident_count' as check_name, COUNT(*) as value,
  CASE WHEN COUNT(*) = 0 THEN '✅ PASS' ELSE '❌ FAIL' END as status
FROM vattenfall_dev.gold.gold_grid_incident_summary
WHERE incident_count < 0

UNION ALL

-- 5. Impossible values: negative duration
SELECT 'Negative duration' as check_name, COUNT(*) as value,
  CASE WHEN COUNT(*) = 0 THEN '✅ PASS' ELSE '❌ FAIL' END as status
FROM vattenfall_dev.gold.gold_grid_incident_summary
WHERE total_duration_minutes < 0

UNION ALL

-- 6. Impossible values: negative customers
SELECT 'Negative customers' as check_name, COUNT(*) as value,
  CASE WHEN COUNT(*) = 0 THEN '✅ PASS' ELSE '❌ FAIL' END as status
FROM vattenfall_dev.gold.gold_grid_incident_summary
WHERE total_customers_affected < 0

UNION ALL

-- 7. Logic check: elevated >= critical (should always be true)
SELECT 'elevated < critical' as check_name, COUNT(*) as value,
  CASE WHEN COUNT(*) = 0 THEN '✅ PASS' ELSE '⚠️ WARNING' END as status
FROM vattenfall_dev.gold.gold_grid_incident_summary
WHERE elevated_incident_count < critical_incident_count;

In [0]:
%sql
-- Sample rows from gold_grid_incident_summary (highest impact incidents)
SELECT 
  event_day,
  region,
  severity_band,
  incident_count,
  elevated_incident_count,
  critical_incident_count,
  total_duration_minutes,
  avg_duration_minutes,
  total_customers_affected,
  avg_customers_per_incident
FROM vattenfall_dev.gold.gold_grid_incident_summary
ORDER BY total_customers_affected DESC
LIMIT 10;

## Validation 4: gold_regional_condition_daily

**Grain:** report_day × region  
**Expected:** 60 rows (15 days × 4 regions)  
**Business Logic:** Composite health score with operational conditions

In [0]:
%sql
-- Table: gold_regional_condition_daily
-- Grain: report_day × region
-- Expected: 60 rows (15 days × 4 regions)

-- 1. Row count
SELECT 'Row Count' as check_name, COUNT(*) as value, 
  CASE WHEN COUNT(*) = 60 THEN '✅ PASS' ELSE '❌ FAIL' END as status
FROM vattenfall_dev.gold.gold_regional_condition_daily

UNION ALL

-- 2. Null dimensions
SELECT 'Null report_day' as check_name, COUNT(*) as value,
  CASE WHEN COUNT(*) = 0 THEN '✅ PASS' ELSE '❌ FAIL' END as status
FROM vattenfall_dev.gold.gold_regional_condition_daily
WHERE report_day IS NULL

UNION ALL

SELECT 'Null region' as check_name, COUNT(*) as value,
  CASE WHEN COUNT(*) = 0 THEN '✅ PASS' ELSE '❌ FAIL' END as status
FROM vattenfall_dev.gold.gold_regional_condition_daily
WHERE region IS NULL

UNION ALL

-- 3. Duplicate grains
SELECT 'Duplicate grains' as check_name, COUNT(*) - COUNT(DISTINCT report_day, region) as value,
  CASE WHEN COUNT(*) = COUNT(DISTINCT report_day, region) THEN '✅ PASS' ELSE '❌ FAIL' END as status
FROM vattenfall_dev.gold.gold_regional_condition_daily

UNION ALL

-- 4. Health score out of range (0-100)
SELECT 'Invalid health_score' as check_name, COUNT(*) as value,
  CASE WHEN COUNT(*) = 0 THEN '✅ PASS' ELSE '❌ FAIL' END as status
FROM vattenfall_dev.gold.gold_regional_condition_daily
WHERE health_score < 0 OR health_score > 100

UNION ALL

-- 5. Negative incident counts
SELECT 'Negative total_incident_count' as check_name, COUNT(*) as value,
  CASE WHEN COUNT(*) = 0 THEN '✅ PASS' ELSE '❌ FAIL' END as status
FROM vattenfall_dev.gold.gold_regional_condition_daily
WHERE total_incident_count < 0

UNION ALL

-- 6. Negative customers affected
SELECT 'Negative total_customers_affected' as check_name, COUNT(*) as value,
  CASE WHEN COUNT(*) = 0 THEN '✅ PASS' ELSE '❌ FAIL' END as status
FROM vattenfall_dev.gold.gold_regional_condition_daily
WHERE total_customers_affected < 0

UNION ALL

-- 7. Operational condition values valid
SELECT 'Invalid operational_condition' as check_name, COUNT(*) as value,
  CASE WHEN COUNT(*) = 0 THEN '✅ PASS' ELSE '❌ FAIL' END as status
FROM vattenfall_dev.gold.gold_regional_condition_daily
WHERE operational_condition NOT IN ('EXCELLENT', 'GOOD', 'FAIR', 'POOR', 'CRITICAL');

In [0]:
%sql
-- Sample rows from gold_regional_condition_daily (worst health scores)
SELECT 
  report_day,
  region,
  health_score,
  operational_condition,
  requires_alert,
  avg_price_eur_mwh,
  weather_risk_score,
  weather_risk_level,
  total_incident_count,
  total_critical_incidents,
  total_customers_affected
FROM vattenfall_dev.gold.gold_regional_condition_daily
ORDER BY health_score ASC
LIMIT 10;

## Final Summary

All 4 gold tables validated with comprehensive checks:

In [0]:
%sql
-- Final summary across all gold tables
SELECT 
  'gold_daily_market_summary' as table_name,
  COUNT(*) as row_count,
  COUNT(DISTINCT report_day) as unique_days,
  COUNT(DISTINCT region) as unique_regions,
  '15 days × 4 regions = 60' as expected
FROM vattenfall_dev.gold.gold_daily_market_summary

UNION ALL

SELECT 
  'gold_weather_impact_summary' as table_name,
  COUNT(*) as row_count,
  COUNT(DISTINCT report_day) as unique_days,
  COUNT(DISTINCT region) as unique_regions,
  '15 days × 4 regions = 60' as expected
FROM vattenfall_dev.gold.gold_weather_impact_summary

UNION ALL

SELECT 
  'gold_grid_incident_summary' as table_name,
  COUNT(*) as row_count,
  COUNT(DISTINCT event_day) as unique_days,
  COUNT(DISTINCT region) as unique_regions,
  'Variable by severity_band' as expected
FROM vattenfall_dev.gold.gold_grid_incident_summary

UNION ALL

SELECT 
  'gold_regional_condition_daily' as table_name,
  COUNT(*) as row_count,
  COUNT(DISTINCT report_day) as unique_days,
  COUNT(DISTINCT region) as unique_regions,
  '15 days × 4 regions = 60' as expected
FROM vattenfall_dev.gold.gold_regional_condition_daily

ORDER BY table_name;

---

## ✅ Gold Layer Validation Complete

### Next Steps

1. **Review Results:** Check all validation cells for ❌ FAIL or ⚠️ WARNING statuses
2. **Investigate Failures:** Query specific rows where validation failed
3. **Fix Data Issues:** Update gold table creation logic if needed
4. **Document:** Record validation results and any data quality issues found
5. **Schedule:** Set up automated validation to run after each gold table refresh

### Business Impact

Gold tables are the **business-facing layer**. Data quality issues here directly affect:
* Executive dashboards
* Operational decisions
* Customer communications
* Regulatory reporting

**Validation is not optional** — it's a critical quality gate before production use.